In [ ]:
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

SEED = 42                      # fixed seed so results can be repeated
np.random.seed(SEED)

# Connect to Google Drive (in Colab). Outside Colab, folders can be set with environment variables.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/capstone'
except ImportError:
    BASE = os.environ.get('CAPSTONE_BASE', './capstone')

RAW = os.environ.get('CAPSTONE_RAW', os.path.join(BASE, 'raw'))
OUT = os.path.join(BASE, 'processed')
os.makedirs(OUT, exist_ok=True)

print('pandas', pd.__version__, '| numpy', np.__version__)
print('Raw folder:', RAW)
print('Output folder:', OUT)

# A simple cleaning log: every step records what happened and how many rows were left.
CLEAN_LOG = []
def log(step, detail, n=None):
    CLEAN_LOG.append({'step': step, 'detail': detail, 'count': n})
    print(f'[{step}] {detail}' + (f': {n:,}' if n is not None else ''))

Mounted at /content/drive
pandas 2.2.3 | numpy 2.4.6
Raw folder: /content/drive/MyDrive/capstone/raw
Output folder: /content/drive/MyDrive/capstone/processed


## Step 1. Find and inventory the raw files

The **year in each name is the start of the academic year**, so `MERGED2018_19_PP` holds Fall 2018 enrollment (`UGDS`), and `MERGED2024_25_PP` holds Fall 2024.

In [ ]:
YEAR_FILES = {
    2013: 'MERGED2013_14_PP.csv',
    2018: 'MERGED2018_19_PP.csv', 2019: 'MERGED2019_20_PP.csv', 2020: 'MERGED2020_21_PP.csv',
    2021: 'MERGED2021_22_PP.csv', 2022: 'MERGED2022_23_PP.csv', 2023: 'MERGED2023_24_PP.csv',
    2024: 'MERGED2024_25_PP.csv',
}
RECENT_FILE = 'Most-Recent-Cohorts-Institution.csv'
HD_FILE = 'hd2018.csv'

def find_file(name):
    # case-insensitive search, so hd2018.csv and HD2018.csv both work
    for f in os.listdir(RAW):
        if f.lower() == name.lower():
            return os.path.join(RAW, f)
    raise FileNotFoundError(f'{name} was not found in {RAW}. Check the file name and folder.')

rows = []
for key, name in list(YEAR_FILES.items()) + [('Most Recent', RECENT_FILE)]:
    path = find_file(name)
    n_cols = len(pd.read_csv(path, nrows=0).columns)
    n_rows = len(pd.read_csv(path, usecols=['UNITID']))     # read one column only to count rows quickly
    rows.append({'file': name, 'rows': n_rows, 'columns': n_cols, 'MB': round(os.path.getsize(path)/1e6, 1)})
hd_path = find_file(HD_FILE)
hd_cols = len(pd.read_csv(hd_path, nrows=0, encoding='latin-1').columns)
hd_rows = len(pd.read_csv(hd_path, usecols=['UNITID'], encoding='latin-1'))
rows.append({'file': HD_FILE, 'rows': hd_rows, 'columns': hd_cols, 'MB': round(os.path.getsize(hd_path)/1e6, 1)})
inventory = pd.DataFrame(rows)
print(inventory.to_string(index=False))

                               file  rows  columns    MB
               MERGED2013_14_PP.csv  7869     3379 104.2
               MERGED2018_19_PP.csv  6807     3379  93.4
               MERGED2019_20_PP.csv  6694     3379  95.1
               MERGED2020_21_PP.csv  6681     3379  76.4
               MERGED2021_22_PP.csv  6543     3379  83.5
               MERGED2022_23_PP.csv  6484     3379  84.5
               MERGED2023_24_PP.csv  6430     3379  82.5
               MERGED2024_25_PP.csv  6322     3379  66.5
Most-Recent-Cohorts-Institution.csv  6278     3379 102.0
                         hd2018.csv  6857       73   4.3


## Step 2. Load only the columns we need

The raw files have 3,379 columns each. We read about 35 of them. `NA` and `PrivacySuppressed` are treated as missing values (they are not zeros).

In [ ]:
ID_COLS = ['UNITID', 'OPEID6', 'INSTNM', 'STABBR', 'MAIN', 'NUMBRANCH', 'PREDDEG', 'ICLEVEL',
           'DISTANCEONLY', 'CONTROL', 'REGION']
FEATURE_COLS = ['ADM_RATE', 'OPENADMP', 'TUITIONFEE_IN', 'NPT4_PUB', 'NPT4_PRIV', 'COSTT4_A',
                'TUITFTE', 'INEXPFTE', 'ENDOWEND', 'AVGFACSAL', 'PCTPELL', 'PCTFLOAN',
                'UGDS_BLACK', 'UGDS_HISP', 'UGDS_ASIAN', 'UGDS_NRA', 'PPTUG_EF',
                'C150_4', 'RET_FT4', 'CDR3', 'MD_EARN_WNE_P7', 'DEBT_MDN']
NA_VALUES = ['NA', 'NULL', 'PrivacySuppressed', 'PS']

def read_cols(name, cols):
    path = find_file(name)
    available = pd.read_csv(path, nrows=0).columns
    use = [c for c in cols if c in available]
    missing = [c for c in cols if c not in available]
    if missing:
        print(f'  note: {name} has no column(s) {missing}')
    return pd.read_csv(path, usecols=use, na_values=NA_VALUES, low_memory=False)

panel_path = os.path.join(OUT, 'campus_panel.csv')
opeyear_path = os.path.join(OUT, 'opeid_by_year.csv')
recent_path = os.path.join(OUT, 'recent_status.csv')

if os.path.exists(panel_path) and os.path.exists(opeyear_path) and os.path.exists(recent_path):
    campus = pd.read_csv(panel_path, low_memory=False)
    ope_year = pd.read_csv(opeyear_path)
    recent = pd.read_csv(recent_path)
    print('Loaded the saved small files from the processed folder:', campus.shape)
else:
    # 2018-19 file: identifiers, filters, and all 2018 characteristics
    campus = read_cols(YEAR_FILES[2018], ID_COLS + FEATURE_COLS + ['UGDS']).rename(columns={'UGDS': 'U2018'})
    ope_parts = [campus[['OPEID6']].dropna().assign(year=2018)]
    for y in [2013, 2019, 2020, 2021, 2022, 2023, 2024]:
        d = read_cols(YEAR_FILES[y], ['UNITID', 'OPEID6', 'UGDS'])
        campus = campus.merge(d[['UNITID', 'UGDS']].rename(columns={'UGDS': f'U{y}'}), on='UNITID', how='left')
        campus[f'in{y}'] = campus['UNITID'].isin(d['UNITID'])          # was this campus in that year's file?
        ope_parts.append(d[['OPEID6']].dropna().assign(year=y))       # which institutions (OPEID6) appear each year
    ope_year = pd.concat(ope_parts).drop_duplicates()
    recent = read_cols(RECENT_FILE, ['UNITID', 'OPEID6', 'CURROPER'])
    campus.to_csv(panel_path, index=False)
    ope_year.to_csv(opeyear_path, index=False)
    recent.to_csv(recent_path, index=False)
    print('Built and saved the small files:', campus.shape)

log('load', 'campus rows in the 2018-19 file', len(campus))

# Institution totals: for each OPEID6 and year, add up UGDS over ALL rows with that OPEID6 (every campus, in the full file)
totals_path = os.path.join(OUT, 'opeid_totals.csv')
if os.path.exists(totals_path):
    ope_totals = pd.read_csv(totals_path)
    print('Loaded the saved institution totals:', ope_totals.shape)
else:
    parts = []
    t18 = campus.dropna(subset=['OPEID6']).groupby('OPEID6')['U2018'].sum(min_count=1)
    parts.append(t18.rename('total').reset_index().assign(year=2018))
    for y in [2013, 2019, 2020, 2021, 2022, 2023, 2024]:
        d = read_cols(YEAR_FILES[y], ['UNITID', 'OPEID6', 'UGDS']).dropna(subset=['OPEID6'])
        t = d.groupby('OPEID6')['UGDS'].sum(min_count=1)
        parts.append(t.rename('total').reset_index().assign(year=y))
    ope_totals = pd.concat(parts)
    ope_totals['OPEID6'] = ope_totals['OPEID6'].astype('int64')
    ope_totals.to_csv(totals_path, index=False)
    print('Built and saved the institution totals:', ope_totals.shape)

Loaded the saved small files from the processed folder: (6807, 48)
[load] campus rows in the 2018-19 file: 6,807
Loaded the saved institution totals: (42015, 3)


## Step 3. Sample filters and the attrition table

Each filter removes some colleges. The table shows how many remain after each step. This table goes in the methods section.

In [ ]:
steps = []
s = campus.copy();                              steps.append(('All institutions in the 2018-19 file', len(s)))
s = s[s['ICLEVEL'] == 1];                       steps.append(('Four-year (ICLEVEL = 1)', len(s)))
s = s[s['PREDDEG'] == 3];                       steps.append(("Bachelor's-predominant (PREDDEG = 3)", len(s)))
s = s[s['CONTROL'].isin([1, 2])];               steps.append(('Public or private nonprofit (CONTROL = 1 or 2)', len(s)))
s = s[s['DISTANCEONLY'] != 1];                  steps.append(('Not online-only (DISTANCEONLY != 1)', len(s)))
s = s[s['U2018'] >= 100];                       steps.append(('At least 100 undergraduates in Fall 2018', len(s)))
s = s[~s['REGION'].isin([0, 9])];               steps.append(('Excluding REGION 0 (service schools) and 9 (outlying areas)', len(s)))
attrition = pd.DataFrame(steps, columns=['step', 'campus rows'])
print(attrition.to_string(index=False))
sample = s.copy()
for st, n in steps: log('filter', st, n)

                                                       step  campus rows
                       All institutions in the 2018-19 file         6807
                                    Four-year (ICLEVEL = 1)         3111
                       Bachelor's-predominant (PREDDEG = 3)         2058
             Public or private nonprofit (CONTROL = 1 or 2)         1883
                        Not online-only (DISTANCEONLY != 1)         1867
                   At least 100 undergraduates in Fall 2018         1750
Excluding REGION 0 (service schools) and 9 (outlying areas)         1701
[filter] All institutions in the 2018-19 file: 6,807
[filter] Four-year (ICLEVEL = 1): 3,111
[filter] Bachelor's-predominant (PREDDEG = 3): 2,058
[filter] Public or private nonprofit (CONTROL = 1 or 2): 1,883
[filter] Not online-only (DISTANCEONLY != 1): 1,867
[filter] At least 100 undergraduates in Fall 2018: 1,750
[filter] Excluding REGION 0 (service schools) and 9 (outlying areas): 1,701


## Step 4. Combine campuses into institutions (OPEID6)

Campuses that share an OPEID6 belong to one institution. Its yearly enrollment is the **sum of all rows with that OPEID6** (every campus, in the full file), so campus openings, closings, and reporting changes do not create false jumps or blanks. The characteristics (location, price, and so on) come from the **main campus**.

In [ ]:
YEARS_ALL = [2013, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
sample = sample.copy()
sample['OPEID6'] = sample['OPEID6'].astype('int64')

# number of rows (campuses) under each OPEID6 in the full 2018-19 file
campus_ok = campus.dropna(subset=['OPEID6']).copy()
campus_ok['OPEID6'] = campus_ok['OPEID6'].astype('int64')
rows_2018 = campus_ok.groupby('OPEID6').size()

tot = ope_totals.pivot(index='OPEID6', columns='year', values='total')       # one row per OPEID6, one column per year

records = []
for ope, g in sample.groupby('OPEID6'):
    main = g.sort_values(['MAIN', 'U2018'], ascending=[False, False]).iloc[0]    # main campus first, then the largest
    rec = main[ID_COLS + FEATURE_COLS].to_dict()
    rec['n_campuses'] = int(rows_2018.get(ope, len(g)))
    rec['multi_campus'] = int(rec['n_campuses'] > 1)
    rec['any_campus_in_2024'] = bool(g['in2024'].any())     # is any 2018 campus of this institution still in the 2024-25 file under its old UNITID?
    for y in YEARS_ALL:
        rec[f'T{y}'] = tot.loc[ope, y] if (ope in tot.index and y in tot.columns) else np.nan
    records.append(rec)
inst = pd.DataFrame(records).rename(columns={'MAIN': 'MAIN_main_campus'})

log('combine', 'institutions after combining campuses', len(inst))
print('Public:', int((inst.CONTROL == 1).sum()), '| Private nonprofit:', int((inst.CONTROL == 2).sum()))
print('Multi-campus institutions:', int(inst.multi_campus.sum()))

[combine] institutions after combining campuses: 1,608
Public: 526 | Private nonprofit: 1082
Multi-campus institutions: 118


## Step 5. Flag possible closures and mergers (exit candidates)

An institution is an **exit candidate** if either:
- **no campus with its OPEID6 appears in the 2024-25 file** (it closed or merged away), or
- **its OPEID6 continues, but none of its 2018 campuses is still in the 2024-25 file under the same UNITID** (the institution was replaced by a new record, as in the 2022 Pennsylvania state university mergers).

These are only candidates. Each one still needs to be verified **munaully** with at least one outside source. Only events effective by Fall 2024 count.

In [ ]:
present_2024 = set(ope_year.loc[ope_year.year == 2024, 'OPEID6'])
last_seen = ope_year.groupby('OPEID6')['year'].max()

inst['opeid_in_2024'] = inst['OPEID6'].isin(present_2024)
inst['exit_candidate'] = (~inst['opeid_in_2024']) | (~inst['any_campus_in_2024'])
inst['exit_reason'] = np.select(
    [~inst['opeid_in_2024'], ~inst['any_campus_in_2024']],
    ['OPEID6 not in the 2024-25 file', 'same OPEID6, but the UNITID was replaced'],
    default='')
inst['last_year_in_files'] = inst['OPEID6'].map(last_seen)

# Operating status of the main campus in the Most Recent file (1 = operating, 0 = closed or not operating)
rec_status = recent.drop_duplicates('UNITID').set_index('UNITID')['CURROPER']
inst['curroper_recent'] = inst['UNITID'].map(rec_status)

print('Exit candidates:', int(inst.exit_candidate.sum()),
      '| public:', int(((inst.CONTROL == 1) & inst.exit_candidate).sum()),
      '| private nonprofit:', int(((inst.CONTROL == 2) & inst.exit_candidate).sum()))
print(inst.loc[inst.exit_candidate, 'exit_reason'].value_counts().to_string())
print('Last year seen in the files (exit candidates):')
print(inst.loc[inst.exit_candidate, 'last_year_in_files'].value_counts().sort_index().to_string())

# A file to fill in by hand while verifying (add the event type, the effective year, and two sources)
to_verify = inst.loc[inst.exit_candidate, ['OPEID6', 'UNITID', 'INSTNM', 'STABBR', 'CONTROL', 'n_campuses',
                                            'T2018', 'last_year_in_files', 'exit_reason', 'curroper_recent']].copy()
for c in ['event_type', 'effective_year', 'partner_or_successor', 'source_1', 'source_2', 'verified_yes_no']:
    to_verify[c] = ''
to_verify.sort_values(['CONTROL', 'INSTNM']).to_csv(os.path.join(OUT, 'exit_candidates_to_verify.csv'), index=False)
log('exit', 'exit candidates (to verify)', int(inst.exit_candidate.sum()))

# ---- Read the hand-verified file (saved in the manual folder) ----
MANUAL = os.environ.get('CAPSTONE_MANUAL', os.path.join(BASE, 'manual'))
ver_path = os.path.join(MANUAL, 'exit_events_verified.csv')
inst['exit_final'] = inst['exit_candidate']            # default: the automatic candidates
inst['exit_event_type'] = ''
inst['exit_effective_year'] = np.nan

if os.path.exists(ver_path):
    ver = pd.read_csv(ver_path, encoding='utf-8-sig', dtype=str, keep_default_na=False)
    need = ['OPEID6', 'event_type', 'effective_year', 'verified_yes_no']
    assert all(c in ver.columns for c in need), f'The verified file needs these columns: {need}'
    ver['OPEID6'] = ver['OPEID6'].astype('int64')
    ver['answer'] = ver['verified_yes_no'].str.strip().str.lower()
    ver['year_num'] = pd.to_numeric(ver['effective_year'], errors='coerce')

    cand = set(inst.loc[inst.exit_candidate, 'OPEID6'])
    print('\nVerified file:', len(ver), 'rows | duplicates:', int(ver.OPEID6.duplicated().sum()),
          '| candidates missing from the file:', len(cand - set(ver.OPEID6)),
          '| rows that are not candidates:', len(set(ver.OPEID6) - cand))
    print('Answers:', ver['answer'].value_counts().to_dict())
    print('Event types:', ver['event_type'].str.strip().str.lower().value_counts().to_dict())
    print('Effective years:', ver['year_num'].value_counts().sort_index().to_dict())

    ok = (ver['answer'] == 'yes') & ver['year_num'].notna() & (ver['year_num'] <= 2024)     # events must be effective by Fall 2024 (every college in the sample was still reporting in 2018)
    odd = ver[~ok]
    if len(odd):
        print('\nRows that will NOT count as exits (answer is not yes, or the year is missing or after 2024):')
        print(odd[['OPEID6', 'INSTNM', 'event_type', 'effective_year', 'verified_yes_no']].to_string(index=False))
    final = ver[ok].set_index('OPEID6')
    inst['exit_final'] = inst['OPEID6'].isin(final.index)
    inst['exit_event_type'] = inst['OPEID6'].map(final['event_type'].str.strip().str.lower()).fillna('')
    inst['exit_effective_year'] = inst['OPEID6'].map(final['year_num'])
    # Note any year that disagrees with the last year the college appears in the files (for your information)
    chk = inst[inst.exit_final & inst.exit_effective_year.notna() & (inst.exit_effective_year < inst.last_year_in_files)]
    if len(chk):
        print('\nEffective year earlier than the last year seen in the files (check the wording in your notes):')
        print(chk[['INSTNM', 'exit_effective_year', 'last_year_in_files']].to_string(index=False))
    print('\nFinal verified exits:', int(inst.exit_final.sum()),
          '| public:', int(((inst.CONTROL == 1) & inst.exit_final).sum()),
          '| private nonprofit:', int(((inst.CONTROL == 2) & inst.exit_final).sum()))
    print(pd.crosstab(inst.loc[inst.exit_final, 'CONTROL'].map({1: 'public', 2: 'private nonprofit'}), inst.loc[inst.exit_final, 'exit_event_type']).to_string())
    log('exit', 'verified exits used in the labels', int(inst.exit_final.sum()))
else:
    print('\nNo verified file found at', ver_path, '- the automatic candidates are used as exits for now.')

Exit candidates: 73 | public: 10 | private nonprofit: 63
exit_reason
OPEID6 not in the 2024-25 file              71
same OPEID6, but the UNITID was replaced     2
Last year seen in the files (exit candidates):
last_year_in_files
2018    13
2019    10
2020     7
2021    10
2022    16
2023    15
2024     2
[exit] exit candidates (to verify): 73

Verified file: 73 rows | duplicates: 0 | candidates missing from the file: 0 | rows that are not candidates: 0
Answers: {'yes': 73}
Event types: {'closure': 48, 'merger': 25}
Effective years: {2017: 1, 2019: 11, 2020: 13, 2021: 6, 2022: 11, 2023: 15, 2024: 16}

Effective year earlier than the last year seen in the files (check the wording in your notes):
                               INSTNM  exit_effective_year  last_year_in_files
       University of Maine at Machias               2017.0                2019
                   Pine Manor College               2020.0                2021
Bloomsburg University of Pennsylvania               2022.0  

## Step 6. Enrollment index and the three labels

The **enrollment index** is a year's enrollment divided by Fall 2018. Rules (5% threshold):
- **Resilient:** the lowest index from 2019 to 2024 is at least 0.95.
- **Recovered:** it fell below 0.95, but the 2024 index is at least 0.95.
- **Vulnerable:** the 2024 index is below 0.95, **or** the institution is a verified exit (`exit_final`).

If a middle year is missing, the label can still be decided when the 2024 index is already below the threshold (vulnerable). If 2024 is at or above the threshold and a year is missing, the label stays blank for you to review. We also make the labels at 10% for the sensitivity check.

In [ ]:
def make_label(row, thr):
    if row['exit_final']:
        return 'vulnerable'
    base = row['T2018']
    if pd.isna(base) or base <= 0:
        return np.nan
    idx = [np.nan if pd.isna(row[f'T{y}']) else row[f'T{y}'] / base for y in range(2019, 2025)]
    last = idx[-1]
    if (not pd.isna(last)) and last < 1 - thr:
        return 'vulnerable'              # still below the threshold in 2024, so a missing middle year cannot change this
    if any(pd.isna(i) for i in idx):
        return np.nan                    # cannot tell resilient from recovered: review these by hand
    return 'resilient' if min(idx) >= 1 - thr else 'recovered'

inst['label_5'] = inst.apply(lambda r: make_label(r, 0.05), axis=1)
inst['label_10'] = inst.apply(lambda r: make_label(r, 0.10), axis=1)
inst['sector'] = inst['CONTROL'].map({1: 'public', 2: 'private nonprofit'})

print('Institutions with a missing label (review):', int(inst.label_5.isna().sum()))
for thr in ['label_5', 'label_10']:
    t = pd.crosstab(inst['sector'], inst[thr], margins=True)
    print(f'\n{thr}  (verified exits are counted as vulnerable)')
    print(t.to_string())
    shares = (inst[thr].value_counts(normalize=True) * 100).round(1)
    print('Shares (%):', shares.to_dict())

Institutions with a missing label (review): 0

label_5  (verified exits are counted as vulnerable)
label_5            recovered  resilient  vulnerable   All
sector                                                   
private nonprofit        174        298         610  1082
public                    61        152         313   526
All                      235        450         923  1608
Shares (%): {'vulnerable': 57.4, 'resilient': 28.0, 'recovered': 14.6}

label_10  (verified exits are counted as vulnerable)
label_10           recovered  resilient  vulnerable   All
sector                                                   
private nonprofit        135        419         528  1082
public                    45        225         256   526
All                      180        644         784  1608
Shares (%): {'vulnerable': 48.8, 'resilient': 40.0, 'recovered': 11.2}


## Step 7. Add locale and Carnegie class (HD2018) and build the derived fields

`LOCALE` and the Carnegie class are blank in the 2018-19 Scorecard file, so they come from the IPEDS 2018 directory (HD2018). Negative codes in HD2018 mean "not available" and become missing values.

In [ ]:
hd = pd.read_csv(find_file(HD_FILE), encoding='latin-1', usecols=['UNITID', 'LOCALE', 'C18BASIC', 'HBCU'], low_memory=False)
for c in ['LOCALE', 'C18BASIC', 'HBCU']:
    hd.loc[hd[c] < 0, c] = np.nan                          # -1, -2, -3 are 'not applicable / not available'
inst = inst.merge(hd, on='UNITID', how='left')
print('Missing after the merge -> LOCALE:', int(inst.LOCALE.isna().sum()),
      '| C18BASIC:', int(inst.C18BASIC.isna().sum()), '| HBCU:', int(inst.HBCU.isna().sum()))

# Derived fields
inst['log_enroll'] = np.log(inst['T2018'])
inst['net_price'] = inst['NPT4_PUB'].fillna(inst['NPT4_PRIV'])                      # public and private net price in one column
inst['grant_aid_intensity'] = (inst['COSTT4_A'] - inst['net_price']) / inst['COSTT4_A']
inst['endow_per_student'] = inst['ENDOWEND'] / inst['T2018']

# Pre-2018 trend: log change in enrollment from Fall 2013 to Fall 2018
ok = inst['T2013'] > 0
inst['trend_13_18_raw'] = np.where(ok, np.log(inst['T2018'] / inst['T2013'].where(ok)), np.nan)
lo, hi = inst['trend_13_18_raw'].quantile([0.01, 0.99])
inst['trend_13_18'] = inst['trend_13_18_raw'].clip(lo, hi)                          # cap extreme values at the 1st and 99th percentile
inst['trend_extreme_flag'] = inst['trend_13_18_raw'].abs() > np.log(2)              # changed by more than a factor of 2

print('Fall 2013 enrollment available:', int(ok.sum()), 'of', len(inst))
print('Trend capped between', round(lo, 2), 'and', round(hi, 2), '| extreme cases to check by hand:', int(inst.trend_extreme_flag.sum()))
inst.loc[inst.trend_extreme_flag, ['OPEID6', 'INSTNM', 'STABBR', 'n_campuses', 'T2013', 'T2018', 'trend_13_18_raw']] \
    .sort_values('trend_13_18_raw').to_csv(os.path.join(OUT, 'trend_extremes_to_check.csv'), index=False)

# Optional hand check: fill the column pre2018_merger_yes_no in a copy of trend_extremes_to_check.csv and save it in the manual folder as trend_extremes_checked.csv
inst['trend_pre2018_merger'] = False
tx_path = os.path.join(MANUAL, 'trend_extremes_checked.csv')
if os.path.exists(tx_path):
    tx = pd.read_csv(tx_path, encoding='utf-8-sig', dtype=str, keep_default_na=False)
    tx['OPEID6'] = tx['OPEID6'].astype('int64')
    yes = set(tx.loc[tx['pre2018_merger_yes_no'].str.strip().str.lower() == 'yes', 'OPEID6'])
    inst['trend_pre2018_merger'] = inst['OPEID6'].isin(yes)
    inst.loc[inst['trend_pre2018_merger'], 'trend_13_18'] = np.nan      # the trend is not meaningful after a merger: leave blank (filled later with the median plus a missing flag)
    print('Trend check file read. Institutions with a merger before 2018 (trend set to blank):', len(yes))
else:
    print('No trend check file yet (optional): manual/trend_extremes_checked.csv')

Missing after the merge -> LOCALE: 0 | C18BASIC: 2 | HBCU: 0
Fall 2013 enrollment available: 1593 of 1608
Trend capped between -0.65 and 0.74 | extreme cases to check by hand: 29
No trend check file yet (optional): manual/trend_extremes_checked.csv


## Step 8. Missing-data audit

Shows the percent missing for each characteristic, overall and by sector. Open-admission colleges have no admission rate (this is meaningful, not an error). Variables that are more than about 40% missing are dropped (for example SAT scores, which were not loaded).

In [ ]:
MODEL_FEATURES = {
    'location':        ['REGION', 'LOCALE'],
    'size':            ['log_enroll', 'C18BASIC', 'CONTROL', 'n_campuses'],
    'selectivity':     ['ADM_RATE', 'OPENADMP'],
    'price':           ['TUITIONFEE_IN', 'net_price', 'grant_aid_intensity'],
    'finances':        ['TUITFTE', 'INEXPFTE', 'endow_per_student', 'AVGFACSAL'],
    'student body':    ['PCTPELL', 'PCTFLOAN', 'UGDS_BLACK', 'UGDS_HISP', 'UGDS_ASIAN', 'UGDS_NRA', 'PPTUG_EF', 'HBCU'],
    'student outcomes':['C150_4', 'RET_FT4', 'CDR3', 'MD_EARN_WNE_P7', 'DEBT_MDN'],
    'pre-2018 trend':  ['trend_13_18'],
}
all_feats = [c for cols in MODEL_FEATURES.values() for c in cols]
overall = (inst[all_feats].isna().mean() * 100).round(1)
by_sector = inst.groupby('sector')[all_feats].apply(lambda d: d.isna().mean() * 100).T.round(1)
audit = pd.concat([overall.rename('overall %'), by_sector], axis=1)
print(audit.to_string())
too_missing = audit.index[audit['overall %'] > 40].tolist()
print('\nVariables more than 40% missing (drop):', too_missing if too_missing else 'none')
audit.to_csv(os.path.join(OUT, 'missing_audit.csv'))

                     overall %  private nonprofit  public
REGION                     0.0                0.0     0.0
LOCALE                     0.0                0.0     0.0
log_enroll                 0.0                0.0     0.0
C18BASIC                   0.1                0.2     0.0
CONTROL                    0.0                0.0     0.0
n_campuses                 0.0                0.0     0.0
ADM_RATE                  10.2               10.5     9.5
OPENADMP                   2.5                2.2     3.2
TUITIONFEE_IN              2.5                2.2     3.2
net_price                  3.7                3.8     3.6
grant_aid_intensity        3.7                3.8     3.6
TUITFTE                    0.1                0.1     0.0
INEXPFTE                   0.1                0.1     0.0
endow_per_student          4.9                6.3     1.9
AVGFACSAL                  0.4                0.6     0.2
PCTPELL                    0.0                0.0     0.0
PCTFLOAN      

## Step 9. Leakage check and save

Every predictor must be known by Fall 2018. The check below makes sure no outcome column (labels, exit flag, enrollment after 2018, or current operating status) is among the predictors.

In [ ]:
FORBIDDEN = ['label_5', 'label_10', 'exit_candidate', 'exit_final', 'exit_event_type', 'exit_effective_year', 'curroper_recent', 'last_year_in_files'] + [f'T{y}' for y in range(2019, 2025)]
leaks = [c for c in all_feats if c in FORBIDDEN]
assert not leaks, f'Leakage: {leaks}'
print('Leakage check passed. Predictors by group:')
for g, cols in MODEL_FEATURES.items():
    print(f'  {g}: {cols}')

# Save the analysis files
keep = (['OPEID6', 'UNITID', 'INSTNM', 'STABBR', 'sector', 'CONTROL', 'n_campuses', 'multi_campus',
         'exit_candidate', 'exit_reason', 'exit_final', 'exit_event_type', 'exit_effective_year', 'trend_pre2018_merger', 'last_year_in_files', 'label_5', 'label_10'] +
        [f'T{y}' for y in YEARS_ALL] + [c for c in all_feats if c not in ('CONTROL', 'n_campuses')] +
        ['trend_13_18_raw', 'trend_extreme_flag'])
keep = list(dict.fromkeys(keep))
inst[keep].to_csv(os.path.join(OUT, 'institutions.csv'), index=False)
attrition.to_csv(os.path.join(OUT, 'attrition_table.csv'), index=False)
pd.DataFrame(CLEAN_LOG).to_csv(os.path.join(OUT, 'cleaning_log.csv'), index=False)

print('\nSaved to', OUT)
print('  institutions.csv   shape:', inst[keep].shape)
print('  also: attrition_table.csv, exit_candidates_to_verify.csv, trend_extremes_to_check.csv, missing_audit.csv, cleaning_log.csv')

Leakage check passed. Predictors by group:
  location: ['REGION', 'LOCALE']
  size: ['log_enroll', 'C18BASIC', 'CONTROL', 'n_campuses']
  selectivity: ['ADM_RATE', 'OPENADMP']
  price: ['TUITIONFEE_IN', 'net_price', 'grant_aid_intensity']
  finances: ['TUITFTE', 'INEXPFTE', 'endow_per_student', 'AVGFACSAL']
  student body: ['PCTPELL', 'PCTFLOAN', 'UGDS_BLACK', 'UGDS_HISP', 'UGDS_ASIAN', 'UGDS_NRA', 'PPTUG_EF', 'HBCU']
  student outcomes: ['C150_4', 'RET_FT4', 'CDR3', 'MD_EARN_WNE_P7', 'DEBT_MDN']
  pre-2018 trend: ['trend_13_18']

Saved to /content/drive/MyDrive/capstone/processed
  institutions.csv   shape: (1608, 54)
  also: attrition_table.csv, exit_candidates_to_verify.csv, trend_extremes_to_check.csv, missing_audit.csv, cleaning_log.csv


## Step 10. Metadata table (raw and derived fields)

Builds the table for the report: each variable's name, description, variable type (role and level of measurement), unit or coding, source, whether it is raw or derived (with the formula), the feature group it belongs to, and the percent missing in the final institution table. It saves the table as CSV, Excel, and HTML, so you can paste it into Word.

The descriptions, roles, levels, and units are written in the list below (edit them there if you want different wording). The percent missing is computed from your data.

In [ ]:
S18 = '2018-19 Scorecard file'
META = [
 # (name, description, role, level, unit or coding, source, raw or derived, group)
 ('OPEID6', 'Federal identifier shared by an institution and its campuses; used to combine campuses', 'ID', 'Nominal', 'Code', 'Scorecard yearly files', 'Raw', 'Identifier'),
 ('UNITID', 'Identifier of the main campus', 'ID', 'Nominal', 'Code', 'Scorecard yearly files', 'Raw', 'Identifier'),
 ('label_5', 'Enrollment trajectory label with a 5% tolerance', 'DV', 'Nominal', 'resilient, recovered, vulnerable', 'Derived from enrollment totals T2018 to T2024', 'Derived: index rule at 5%; exit candidates count as vulnerable', 'Outcome'),
 ('label_10', 'Enrollment trajectory label with a 10% tolerance (sensitivity check)', 'DV', 'Nominal', 'resilient, recovered, vulnerable', 'Derived from enrollment totals T2018 to T2024', 'Derived: same rule at 10%', 'Outcome'),
 ('exit_final', 'Verified closure or merger effective by Fall 2024', 'DV (secondary)', 'Nominal', 'True or False', 'Automatic candidates (OPEID6 and UNITID in the 2024-25 file), verified by hand with outside sources', 'Derived: exit rules plus manual verification', 'Outcome'),
 ('exit_event_type', 'Type of verified exit event (blank if no exit)', 'DV (secondary)', 'Nominal', 'closure, merger', 'Hand-verified file (outside sources)', 'Derived: manual coding', 'Outcome'),
 ('exit_effective_year', 'Year the exit became effective (blank if no exit)', 'DV (secondary)', 'Interval', 'Calendar year', 'Hand-verified file (outside sources)', 'Derived: manual coding', 'Outcome'),
 ('T2013 to T2024', 'Total undergraduate enrollment of the institution, all campuses, each fall (2013 and 2018 to 2024); blank for years after an exit', 'Outcome input', 'Ratio', 'Students (degree-seeking undergraduates)', 'Scorecard yearly files (UGDS, IPEDS)', 'Derived: sum of UGDS over all rows with the OPEID6', 'Outcome input'),
 ('REGION', 'Region of the college (BEA regions)', 'IV', 'Nominal', 'Codes 1 to 8 (0 and 9 excluded)', S18 + ' (IPEDS)', 'Raw', 'Location'),
 ('LOCALE', 'Type of area where the college is located (city, suburb, town, rural)', 'IV', 'Nominal', 'Codes 11 to 43', 'IPEDS HD2018', 'Raw', 'Location'),
 ('log_enroll', 'Log of total undergraduate enrollment in Fall 2018', 'IV', 'Ratio', 'Log of students', 'Derived from T2018', 'Derived: ln(T2018)', 'Size'),
 ('C18BASIC', 'Carnegie basic classification (2018)', 'IV', 'Nominal', 'Carnegie codes', 'IPEDS HD2018', 'Raw', 'Size'),
 ('CONTROL', 'Sector of the college', 'IV', 'Nominal', '1 = public, 2 = private nonprofit', S18 + ' (IPEDS)', 'Raw', 'Size'),
 ('n_campuses', 'Number of campus rows under the OPEID6', 'IV', 'Ratio', 'Campuses', S18 + ' (IPEDS)', 'Derived: count of rows per OPEID6', 'Size'),
 ('ADM_RATE', 'Admission rate', 'IV', 'Ratio', 'Proportion, 0 to 1 (blank for open admission)', S18 + ' (IPEDS)', 'Raw', 'Selectivity'),
 ('OPENADMP', 'Open admissions policy indicator', 'IV', 'Nominal', '1 = yes, 2 = no, 3 = does not enroll first-time students', S18 + ' (IPEDS)', 'Raw', 'Selectivity'),
 ('TUITIONFEE_IN', 'In-state tuition and fees', 'IV', 'Ratio', 'USD per year', S18 + ' (IPEDS)', 'Raw', 'Price'),
 ('net_price', 'Average net price (cost of attendance minus grant aid) for students receiving federal aid', 'IV', 'Ratio', 'USD per year', S18 + ' (IPEDS: NPT4_PUB for public, NPT4_PRIV for private)', 'Derived: NPT4_PUB, else NPT4_PRIV', 'Price'),
 ('COSTT4_A', 'Average cost of attendance (input to grant-aid intensity)', 'Input', 'Ratio', 'USD per year', S18 + ' (IPEDS)', 'Raw', 'Price'),
 ('grant_aid_intensity', 'Share of the cost of attendance covered by aid', 'IV', 'Ratio', 'Proportion', 'Derived from COSTT4_A and net price', 'Derived: (COSTT4_A minus net price) divided by COSTT4_A', 'Price'),
 ('TUITFTE', 'Net tuition revenue per full-time-equivalent student', 'IV', 'Ratio', 'USD per FTE student', S18 + ' (IPEDS)', 'Raw', 'Finances'),
 ('INEXPFTE', 'Instructional expenditures per full-time-equivalent student', 'IV', 'Ratio', 'USD per FTE student', S18 + ' (IPEDS)', 'Raw', 'Finances'),
 ('ENDOWEND', 'Endowment value at the end of the fiscal year (input to endowment per student)', 'Input', 'Ratio', 'USD', S18 + ' (IPEDS)', 'Raw', 'Finances'),
 ('endow_per_student', 'Endowment per undergraduate student', 'IV', 'Ratio', 'USD per student', 'Derived from ENDOWEND and T2018', 'Derived: ENDOWEND divided by T2018', 'Finances'),
 ('AVGFACSAL', 'Average faculty salary', 'IV', 'Ratio', 'USD (check the pay period in the dictionary)', S18 + ' (IPEDS)', 'Raw', 'Finances'),
 ('PCTPELL', 'Share of undergraduates who receive a Pell Grant (low-income proxy)', 'IV', 'Ratio', 'Proportion, 0 to 1', S18 + ' (IPEDS)', 'Raw', 'Student body'),
 ('PCTFLOAN', 'Share of undergraduates who receive a federal student loan', 'IV', 'Ratio', 'Proportion, 0 to 1', S18 + ' (IPEDS)', 'Raw', 'Student body'),
 ('UGDS_BLACK', 'Share of undergraduates who are Black', 'IV', 'Ratio', 'Proportion, 0 to 1', S18 + ' (IPEDS)', 'Raw', 'Student body'),
 ('UGDS_HISP', 'Share of undergraduates who are Hispanic', 'IV', 'Ratio', 'Proportion, 0 to 1', S18 + ' (IPEDS)', 'Raw', 'Student body'),
 ('UGDS_ASIAN', 'Share of undergraduates who are Asian', 'IV', 'Ratio', 'Proportion, 0 to 1', S18 + ' (IPEDS)', 'Raw', 'Student body'),
 ('UGDS_NRA', 'Share of undergraduates who are non-resident aliens', 'IV', 'Ratio', 'Proportion, 0 to 1', S18 + ' (IPEDS)', 'Raw', 'Student body'),
 ('PPTUG_EF', 'Share of undergraduates who are part-time', 'IV', 'Ratio', 'Proportion, 0 to 1', S18 + ' (IPEDS)', 'Raw', 'Student body'),
 ('HBCU', 'Historically Black college or university', 'IV', 'Nominal', '1 = yes, 2 = no', 'IPEDS HD2018', 'Raw', 'Student body'),
 ('C150_4', 'Completion rate for first-time, full-time students within 150% of expected time', 'IV', 'Ratio', 'Proportion, 0 to 1', S18 + ' (IPEDS)', 'Raw', 'Student outcomes'),
 ('RET_FT4', 'First-year retention rate for first-time, full-time students', 'IV', 'Ratio', 'Proportion, 0 to 1', S18 + ' (IPEDS)', 'Raw', 'Student outcomes'),
 ('CDR3', 'Three-year cohort default rate', 'IV', 'Ratio', 'Proportion, 0 to 1', S18 + ' (FSA)', 'Raw', 'Student outcomes'),
 ('MD_EARN_WNE_P7', 'Median earnings of students working and not enrolled 7 years after entry', 'IV', 'Ratio', 'USD', S18 + ' (Treasury)', 'Raw', 'Student outcomes'),
 ('DEBT_MDN', 'Median loan principal when entering repayment', 'IV', 'Ratio', 'USD', S18 + ' (NSLDS)', 'Raw', 'Student outcomes'),
 ('trend_13_18', 'Enrollment trend before 2018: log change from Fall 2013 to Fall 2018, capped at the 1st and 99th percentiles', 'IV', 'Ratio', 'Log change', '2013-14 and 2018-19 Scorecard files (UGDS)', 'Derived: ln(T2018 divided by T2013), capped; blank after a merger before 2018 (if checked)', 'Pre-2018 trend'),
]

# Make sure every predictor used in the models has a row in the table
listed = {m[0] for m in META}
absent = [c for c in all_feats if c not in listed]
assert not absent, f'Add these predictors to META: {absent}'

def pct_missing(name, frame=None):
    d = inst if frame is None else frame
    if name in ('exit_event_type', 'exit_effective_year'):
        return 'n/a (blank unless exit)'
    if name == 'T2013 to T2024':
        return round(d[[f'T{y}' for y in YEARS_ALL]].isna().mean().max() * 100, 1)
    return round(d[name].isna().mean() * 100, 1)

rows = []
for name, desc, role, level, unit, source, derived, group in META:
    rows.append({'Name': name, 'Description': desc, 'Variable type': f'{role}; {level.lower()}',
                 'Unit or coding': unit, 'Source': source, 'Raw or derived': derived,
                 'Feature group': group, 'Missing (%)': pct_missing(name)})
meta_df = pd.DataFrame(rows)
print(meta_df.to_string(index=False))
print('\nRows:', len(meta_df), '| derived fields:', int(meta_df['Raw or derived'].str.startswith('Derived').sum()))

# Extended version: adds percent missing by sector and the official dictionary name, when the dictionary file is in the raw folder
ext = meta_df.copy()
for sec in ['public', 'private nonprofit']:
    sub = inst[inst['sector'] == sec]
    ext[f'Missing, {sec} (%)'] = [pct_missing(m[0], sub) for m in META]
try:
    dd = pd.read_excel(find_file('CollegeScorecardDataDictionary.xlsx'), sheet_name='Institution_Data_Dictionary')
    dd = dd.dropna(subset=['VARIABLE NAME']).drop_duplicates('VARIABLE NAME').set_index('VARIABLE NAME')['NAME OF DATA ELEMENT']
    ext['Official dictionary name'] = [dd.get(m[0], '') for m in META]
except Exception as e:
    print('Dictionary file not used:', e)

# Save: CSV, Excel, and HTML
meta_df.to_csv(os.path.join(OUT, 'metadata_table.csv'), index=False)
try:
    with pd.ExcelWriter(os.path.join(OUT, 'metadata_table.xlsx'), engine='openpyxl') as writer:
        meta_df.to_excel(writer, sheet_name='Metadata table', index=False)
        ext.to_excel(writer, sheet_name='Extended', index=False)
        for ws in writer.sheets.values():
            for col_cells in ws.columns:
                width = min(60, max(len(str(c.value)) if c.value is not None else 0 for c in col_cells) + 2)
                ws.column_dimensions[col_cells[0].column_letter].width = width
    print('Saved metadata_table.xlsx (sheets: Metadata table, Extended)')
except ImportError:
    print('Excel file not saved (openpyxl is not installed); use the CSV or HTML file.')
style = ('<style>table{border-collapse:collapse;font-family:Arial;font-size:10pt}'
         'th,td{border:1px solid #999;padding:3px 6px;vertical-align:top}th{background:#D9E2F3}</style>')
with open(os.path.join(OUT, 'metadata_table.html'), 'w', encoding='utf-8') as f:
    f.write(style + meta_df.to_html(index=False, escape=True))
print('Saved metadata_table.csv and metadata_table.html in', OUT)

               Name                                                                                                                       Description            Variable type                                           Unit or coding                                                                                              Source                                                                             Raw or derived    Feature group             Missing (%)
             OPEID6                                            Federal identifier shared by an institution and its campuses; used to combine campuses              ID; nominal                                                     Code                                                                              Scorecard yearly files                                                                                        Raw       Identifier                     0.0
             UNITID                                                     